In [1]:
import os
from functools import partial
from scipy import constants
import joblib
import sys
import jax
import jax.numpy as jnp
import numpy as np
import optax
from jax import config
from numpy.polynomial.hermite import hermgauss
import os 
from flows.basis.basis import Basis
#from flows.basis.hermite_custom_jvp import hermite
#from flows.hamiltonian import hamiltonian, hamiltonian_trace, eigenvalues
#from flows.models.models import Linear
import matplotlib.pyplot as plt 
from numpy.polynomial.hermite import hermgauss
from flows.Bases import Hermite 
from flows.types import *
from flows.bases import *
from flows.models.linear import Linear
import math

## Set up potential and KEO 

In [2]:
mO, mH = [15.994914619257,1.00782503223]
m1 = mH
m2 = mO
a_m,D_m,x0_m = 2.1440, 42301, 0.0
# Number of states to compute and optimize
no_states = 23

# Potential energy surface
def potential(x):
    return (D_m * (1.0 - jnp.exp(-a_m * (x - x0_m)))**2)[:,0]#.reshape(-1,1)

G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

def abs_det_jac_x(model, params, x):
    def det(params):
        def det(x):
            return jnp.abs(jnp.linalg.det(jax.jacrev(model.apply, 1)(params, x)))
        return jax.vmap(det, in_axes=0)(x)
    return jax.jit(det)(params)
# G-matrix for the Morse potential
def gmat(x):
    npoints = x.shape[0]
    gvib = (m1+m2)/(m1*m2) * jax.numpy.ones_like(x)[:,:,None] * G_to_invcm
    grot = jnp.zeros((npoints,))
    gcor = jnp.zeros((npoints, 1))
    return gvib, grot, gcor



## Set up basis and quadrature grid 

In [3]:

x, w = hermgauss(100)
x = x.reshape(-1,1)


## Set up a fixed linear flow

In [4]:
a = jnp.array([0.08222986])
b = jnp.array([0.])
print(f"Physical params are a: {a}, b: {b}")
model = Linear(a=a, b=b)
params = model.init(jax.random.PRNGKey(0), x[:,0].reshape(-1,1))

model_r = lambda params, x: model.apply(params, x[:,0].reshape(-1,1), mode="direct")
 
model_x = lambda params, x: (
    model.apply(params, x[:, 0].reshape(-1, 1), mode="inverse")
    if x.ndim == 2 else
    model.apply(params, jnp.array(x[0]).reshape(1, -1), mode="inverse")[0, :]
        )
r = model_r(params, x)
print(r.shape)
rmin = np.min(r, axis=0)
rmax = np.max(r, axis=0)
print("Min and max values of physical coords:\n", rmin, rmax)

def _grad_log_abs_det_jac_x(model, params, x_batch, **kwargs):
    def det(x):
        return jnp.log(
            jnp.abs(jnp.linalg.det(jax.jacrev(model, argnums=1)(params, x, **kwargs)))
        )

    return jax.vmap(jax.grad(det), in_axes=0)(x_batch)

def _jac_x(model, params, x_batch, **kwargs):
    def jac(x):
        return jax.jacrev(model, argnums=1)(params, x, **kwargs)

    return jax.vmap(jac, in_axes=0)(x_batch)



Physical params are a: [0.08222986], b: [0.]
(100, 1)
Min and max values of physical coords:
 [-1.10241358] [1.10241358]


## Set up Hamiltonian and compute energies

In [5]:
def hamiltonian(params, x, psi1, dpsi1, psi2, dpsi2, w):
    r = model_r(params, x)
    p = potential(r)
    gvib, _, _ = gmat(r)
    dlog_det = _grad_log_abs_det_jac_x(model_x, params, r)
    
    df = _jac_x(model_x, params, r)
    gvib1 = jnp.einsum("gka,gab...,glb->gkl...", df, gvib, df)
    gvib2 = 0.5 * jnp.einsum("gka,gab...,gb->gk...", df, gvib, dlog_det)
    gvib3 = 0.5 * jnp.einsum("ga,gab...,gkb->gk...", dlog_det, gvib, df)
    gvib4 = 0.25 * jnp.einsum("ga,gab...,gb->g...", dlog_det, gvib, dlog_det)

    keo_vib = (
            jnp.einsum("gik,gkl...,gjl,g->ij...", dpsi1, gvib1, dpsi2, w)
            + jnp.einsum("gik,gk...,gj,g->ij...", dpsi1, gvib2, psi2, w)
            + jnp.einsum("gi,gk...,gjk,g->ij...", psi1, gvib3, dpsi2, w)
            + jnp.einsum("gi,g...,gj,g->ij...", psi1, gvib4, psi2, w)
        )
    poten = jnp.einsum("gi,gj,g...,g->ij...", psi1, psi2, p, w)
    ham = 0.5 * (keo_vib ) + poten
    return ham

def eigensolve(par, psi1, dpsi1, psi2, dpsi2, no_states):
    h = hamiltonian(par, x, psi1, dpsi1, psi2, dpsi2, w)
    e, _ = jax.numpy.linalg.eigh(h)
    return e[:no_states]

eigensolve_jit = partial(jax.jit, static_argnums=(5))(eigensolve)
    

## Perform energy calculations for different values of N

In [6]:
N_values = np.arange(22, 32, 2)
e_values = []
for nmax in N_values:
    
    n_basis = [nmax for _ in range(x.shape[1])]
    w_basis = [1 for _ in range(x.shape[1])]
    # Note that when we project we need to multiply by the weights of the basis
    basis_r = Hermite.init_basis(n_basis, w_basis, nmax, orthotype=orthoType.ortho) # 

    psi_o = partial(Hermite.batch_basis_values, basis_r)  
        
    dpsi_o = partial(Hermite.batch_dbasis_values, basis_r)
    _psi = psi_o(x)
    _dpsi = dpsi_o(x)
    print(type(_psi), _psi.shape)
    print("First few eigenvalues on a test set using initial params:")
    e = eigensolve_jit(params, _psi, _dpsi, _psi, _dpsi, no_states)
    e_values.append(e)
np.savez(f"simulations_data/Morse_Hermite.npz", N_values=N_values, e_values=e_values)

<class 'jaxlib._jax.ArrayImpl'> (100, 23)
First few eigenvalues on a test set using initial params:
<class 'jaxlib._jax.ArrayImpl'> (100, 25)
First few eigenvalues on a test set using initial params:
<class 'jaxlib._jax.ArrayImpl'> (100, 27)
First few eigenvalues on a test set using initial params:
<class 'jaxlib._jax.ArrayImpl'> (100, 29)
First few eigenvalues on a test set using initial params:
<class 'jaxlib._jax.ArrayImpl'> (100, 31)
First few eigenvalues on a test set using initial params:
